# Perceptrón simple y MLP (versión modificada)
Cambios principales: más compuertas lógicas, pesos iniciales aleatorios, comparación de tasas de aprendizaje y de neuronas ocultas, `make_moons` con más ruido, Dropout + L2 + EarlyStopping, métricas extra (F1, ROC/AUC) y comparación con modelos clásicos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, roc_curve, auc, accuracy_score)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = (7, 5)
SEMILLA = 7
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)
print("TensorFlow:", tf.__version__)

## 1. Perceptrón simple (pesos iniciales aleatorios)

In [ ]:
class PerceptronSimple:
    def __init__(self, n_entradas, tasa_aprendizaje=0.1, semilla=SEMILLA):
        rng = np.random.default_rng(semilla)
        self.pesos = rng.uniform(-0.1, 0.1, n_entradas)   # antes: ceros
        self.sesgo = 0.0
        self.tasa_aprendizaje = tasa_aprendizaje

    def activacion(self, z):
        return np.where(z >= 0, 1, 0)

    def predecir(self, x):
        return int(self.activacion(np.dot(x, self.pesos) + self.sesgo))

    def predecir_lote(self, X):
        return self.activacion(X @ self.pesos + self.sesgo)

    def entrenar(self, X, y, epocas=20):
        historial_errores = []
        for _ in range(epocas):
            errores = 0
            for xi, yi in zip(X, y):
                error = yi - self.predecir(xi)
                if error != 0:
                    self.pesos += self.tasa_aprendizaje * error * xi
                    self.sesgo += self.tasa_aprendizaje * error
                    errores += 1
            historial_errores.append(errores)
        return historial_errores


def graficar_frontera(predictor, X, y, titulo, ax=None, rango=(-0.5, 1.5)):
    """predictor: función que recibe un arreglo (n,2) y devuelve predicciones/probabilidades."""
    if ax is None:
        _, ax = plt.subplots()
    xx, yy = np.meshgrid(np.linspace(*rango, 200), np.linspace(*rango, 200))
    Z = np.asarray(predictor(np.c_[xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=20, alpha=0.35, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=130)
    ax.set_title(titulo)
    ax.set_xlabel("x1"); ax.set_ylabel("x2")

## 2. Compuertas AND, OR, NAND y NOR (todas linealmente separables)

In [ ]:
X_log = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
compuertas = {
    "AND":  np.array([0, 0, 0, 1]),
    "OR":   np.array([0, 1, 1, 1]),
    "NAND": np.array([1, 1, 1, 0]),
    "NOR":  np.array([1, 0, 0, 0]),
}

fig, ejes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (nombre, y_c) in zip(ejes.ravel(), compuertas.items()):
    p = PerceptronSimple(2, tasa_aprendizaje=0.1)
    errs = p.entrenar(X_log, y_c, epocas=15)
    print(f"{nombre}: errores por época = {errs} | pesos = {np.round(p.pesos, 2)}, sesgo = {p.sesgo:.2f}")
    graficar_frontera(p.predecir_lote, X_log, y_c, f"Perceptrón — {nombre}", ax=ax)
plt.tight_layout()
plt.show()

## 3. Efecto de la tasa de aprendizaje (compuerta OR)

In [ ]:
for tasa in [0.01, 0.1, 0.5, 1.0]:
    p = PerceptronSimple(2, tasa_aprendizaje=tasa)
    errs = p.entrenar(X_log, compuertas["OR"], epocas=50)
    conv = next((i + 1 for i, e in enumerate(errs) if e == 0), None)
    print(f"tasa = {tasa:<5} -> converge en la época: {conv}")

## 4. XOR: el perceptrón simple falla

In [ ]:
y_xor = np.array([0, 1, 1, 0])
p_xor = PerceptronSimple(2, tasa_aprendizaje=0.1)
errs_xor = p_xor.entrenar(X_log, y_xor, epocas=30)

fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
ejes[0].plot(range(1, 31), errs_xor, marker="s", color="darkorange")
ejes[0].set_title("Errores por época en XOR (no converge)")
ejes[0].set_xlabel("Época"); ejes[0].set_ylabel("N.º de errores")
graficar_frontera(p_xor.predecir_lote, X_log, y_xor, "Perceptrón — XOR (falla)", ax=ejes[1])
plt.tight_layout()
plt.show()

## 5. MLP para XOR: ¿cuántas neuronas ocultas hacen falta?

In [ ]:
X_f = X_log.astype("float32")
y_f = y_xor.astype("float32")

def crear_mlp_xor(n_ocultas, activacion="tanh"):
    m = keras.Sequential([
        layers.Input(shape=(2,)),
        layers.Dense(n_ocultas, activation=activacion),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(0.05), loss="binary_crossentropy", metrics=["accuracy"])
    return m

modelos_xor = {}
for n in [2, 3, 8]:
    keras.backend.clear_session()
    tf.random.set_seed(SEMILLA)
    m = crear_mlp_xor(n)
    h = m.fit(X_f, y_f, epochs=400, verbose=0)
    modelos_xor[n] = m
    print(f"{n} neuronas ocultas -> precisión final = {h.history['accuracy'][-1]:.2f} | pérdida = {h.history['loss'][-1]:.4f}")

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (n, m) in zip(ejes, modelos_xor.items()):
    graficar_frontera(lambda g, m=m: m.predict(g.astype("float32"), verbose=0).ravel(),
                      X_log, y_xor, f"MLP XOR — {n} neuronas (tanh)", ax=ax)
plt.tight_layout()
plt.show()

## 6. Funciones de activación (se agrega Leaky ReLU)

In [ ]:
z = np.linspace(-6, 6, 300)
funciones = {
    "Sigmoide": 1 / (1 + np.exp(-z)),
    "Tanh": np.tanh(z),
    "ReLU": np.maximum(0, z),
    "Leaky ReLU": np.where(z > 0, z, 0.1 * z),
}
fig, ejes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, (nombre, f) in zip(ejes, funciones.items()):
    ax.plot(z, f, linewidth=2.5, color="#7B2D8E")
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(nombre); ax.set_xlabel("z")
plt.tight_layout()
plt.show()

## 7. Dataset make_moons con más ruido

In [ ]:
X, y = make_moons(n_samples=800, noise=0.30, random_state=SEMILLA)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEMILLA, stratify=y)

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.transform(X_test)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=30)
plt.title("make_moons (noise = 0.30) — no separable con una recta")
plt.show()

## 8. MLP con Dropout, regularización L2 y EarlyStopping

In [ ]:
keras.backend.clear_session()
tf.random.set_seed(SEMILLA)

modelo = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-3)),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu", kernel_regularizer=regularizers.l2(1e-3)),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer=keras.optimizers.Adam(0.005), loss="binary_crossentropy", metrics=["accuracy"])

parada = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
hist = modelo.fit(X_train_esc, y_train, epochs=200, batch_size=32,
                  validation_split=0.2, callbacks=[parada], verbose=0)
print("Épocas realmente entrenadas:", len(hist.history["loss"]))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for ax, met, tit in zip(ejes, ["loss", "accuracy"], ["Pérdida", "Precisión"]):
    ax.plot(hist.history[met], label="entrenamiento")
    ax.plot(hist.history["val_" + met], label="validación")
    ax.set_title(tit); ax.set_xlabel("Época"); ax.legend()
plt.tight_layout()
plt.show()

## 9. Evaluación: frontera, matriz de confusión, F1 y curva ROC

In [ ]:
prob_test = modelo.predict(X_test_esc, verbose=0).ravel()
y_pred = (prob_test > 0.5).astype(int)

fig, ejes = plt.subplots(1, 3, figsize=(18, 5))

# Frontera
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                     np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
Z = modelo.predict(escalador.transform(np.c_[xx.ravel(), yy.ravel()]), verbose=0).reshape(xx.shape)
ejes[0].contourf(xx, yy, Z, levels=20, cmap="coolwarm", alpha=0.6)
ejes[0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", edgecolor="k", s=40)
ejes[0].set_title("Frontera de decisión (test)")

# Matriz de confusión
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(ax=ejes[1], cmap="Greens", colorbar=False)
ejes[1].set_title("Matriz de confusión")

# ROC
fpr, tpr, _ = roc_curve(y_test, prob_test)
ejes[2].plot(fpr, tpr, lw=2.5, label=f"AUC = {auc(fpr, tpr):.3f}")
ejes[2].plot([0, 1], [0, 1], "--", color="gray")
ejes[2].set_title("Curva ROC"); ejes[2].set_xlabel("FPR"); ejes[2].set_ylabel("TPR"); ejes[2].legend()
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=["Clase 0", "Clase 1"]))

## 10. Comparación con modelos clásicos

In [ ]:
resultados = {"MLP (Keras)": accuracy_score(y_test, y_pred)}
for nombre, clf in {
    "Regresión logística": LogisticRegression(),
    "SVM (kernel RBF)": SVC(kernel="rbf", C=1.0),
}.items():
    clf.fit(X_train_esc, y_train)
    resultados[nombre] = accuracy_score(y_test, clf.predict(X_test_esc))

plt.bar(resultados.keys(), resultados.values(), color=["#7B2D8E", "#E07B39", "#2E86AB"])
plt.ylim(0.5, 1.0)
plt.ylabel("Precisión en test")
plt.title("Comparación de modelos en make_moons")
for i, v in enumerate(resultados.values()):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.show()

## 11. Extra: el mismo MLP sobre make_circles

In [ ]:
Xc, yc = make_circles(n_samples=600, noise=0.1, factor=0.4, random_state=SEMILLA)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=SEMILLA, stratify=yc)

keras.backend.clear_session()
m_c = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
m_c.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
m_c.fit(Xc_tr, yc_tr, epochs=80, batch_size=16, verbose=0)
print(f"Precisión en test (make_circles): {m_c.evaluate(Xc_te, yc_te, verbose=0)[1]:.3f}")

graficar_frontera(lambda g: m_c.predict(g, verbose=0).ravel(), Xc_te, yc_te,
                  "MLP sobre make_circles", rango=(-1.5, 1.5))
plt.show()